# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahmoud-mos/my-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method Choice: Decision Tree Classifier 

Classical machine learning is the recommended approach for finding predictive patterns in structured data.  Decision trees can capture non-linear rules without becoming a black box.  They provide strong interpretability through feature importance and tree paths, which builds trust when transitioning away from hardcoded heuristics.  The goal is not to maximize complexity, as an overly flexible model risks overfitting. 

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Verify Decision Tree Classifier setup

from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=4, random_state=42)
print("Model initialized:", model)


Model initialized: DecisionTreeClassifier(max_depth=4, random_state=42)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

##Split Design: Time-Based Split

A random split is dangerous here because examples (like web pages tracked over multiple days) are not perfectly independent.  Using a random split when independence is violated often leaks entity-specific or time-specific patterns into the test set, artificially inflating metrics.  A time-based split explicitly tests whether the model can generalize to future predictions, making it the most honest and realistic evaluation.

##Split Design: Chronological (Time-Based) SplitA random split violates independence because identical content pages are tracked across consecutive dates.  Random splitting causes temporal leakage, artificially inflating test set metrics.  Sorting chronologically and splitting the earliest 80% for training and latest 20% for testing provides an honest evaluation of future generalization.  

In [17]:
import os
import duckdb
import pandas as pd
from dotenv import load_dotenv

# 1. Load Hugging Face token from .env
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

con = duckdb.connect()

# 2. Register native Hugging Face secret (prevents connection drops & 401s)
con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

# 3. Query dataset with Hive partitioning enabled
rel = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

query = f"""
SELECT 
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    scroll_events,
    (gsc_sum_position / (gsc_impressions + 0.001)) as avg_position,
    CASE WHEN gsc_impressions > 200 AND gsc_clicks = 0 THEN 1 ELSE 0 END as target_needs_review
FROM read_parquet('{rel}', hive_partitioning=true)
WHERE gsc_data_available = TRUE;
"""
df = con.sql(query).df().fillna(0)

# 4. Sequential 80/20 Split
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:].copy()

print(f"Dataset successfully loaded: {len(df)} total rows")
print(f"Train split: {len(train_df)} rows | Test split: {len(test_df)} rows")

Dataset successfully loaded: 3611061 total rows
Train split: 2888848 rows | Test split: 722213 rows


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Evaluated on the exact same test split using identical features (gsc_impressions, scroll_events, avg_position). The ML model is benchmarked against the Week 4 baseline rule (gsc_impressions > 500 and scroll_events == 0).

In [18]:
from sklearn.metrics import precision_score, recall_score, f1_score

features = ['gsc_impressions', 'scroll_events', 'avg_position']
X_train, y_train = train_df[features], train_df['target_needs_review']
X_test, y_test = test_df[features], test_df['target_needs_review']

# 1. Week 4 Baseline Rule Predictions
test_df['baseline_pred'] = ((test_df['gsc_impressions'] > 500) & (
    test_df['scroll_events'] == 0)).astype(int)

# 2. Week 5 Decision Tree Model
model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)
test_df['model_pred'] = model.predict(X_test)

# 3. Model vs Baseline Comparison Table
results = pd.DataFrame({
    'System': ['Baseline Rule (Week 4)', 'Decision Tree (Week 5)'],
    'Precision': [
        precision_score(y_test, test_df['baseline_pred'], zero_division=0),
        precision_score(y_test, test_df['model_pred'], zero_division=0)
    ],
    'Recall': [
        recall_score(y_test, test_df['baseline_pred'], zero_division=0),
        recall_score(y_test, test_df['model_pred'], zero_division=0)
    ],
    'F1 Score': [
        f1_score(y_test, test_df['baseline_pred'], zero_division=0),
        f1_score(y_test, test_df['model_pred'], zero_division=0)
    ]
})

# Replace this line:
# print(results.to_markdown(index=False))

print(results.to_string(index=False))

                System  Precision   Recall  F1 Score
Baseline Rule (Week 4)   0.378681 0.198883  0.260796
Decision Tree (Week 5)   0.594325 0.804660  0.683681


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Error Analysis & Interpretability

Feature Reliance: Feature importances show gsc_impressions and avg_position drive the majority of split decisions.  False Positives: The model flags navigation or brand pages that receive search impressions but zero clicks because users answered their query directly on the SERP snippet.  False Negatives: Pages with lower impression counts (~150-190) that drop off in clicks are missed due to the strict impression boundary learned by the tree.Decision Value: Improving offline precision does not automatically equal business value if the model flags intentional low-click pages. 

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Feature Importance Analysis
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("--- Feature Importances ---")
print(importance_df.to_string(index=False))

--- Feature Importances ---
        Feature  Importance
gsc_impressions    0.933280
   avg_position    0.050895
  scroll_events    0.015825


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.